# Smart Energy Consumption Predictor
## Exploratory Data Analysis & Model Training Notebook

This notebook walks through:
1. Dataset loading and exploration
2. Data cleaning and feature engineering
3. Visualisation of consumption trends
4. Training LSTM and Linear Regression models
5. Comparing model performance
6. Anomaly detection demo
7. Generating optimisation suggestions

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (12, 5)})
print('Libraries loaded ✓')

## 1. Load & Explore Dataset

In [ ]:
from models.data_loader import load_raw, clean, engineer_features, resample_hourly

# Use synthetic=True when UCI file is not downloaded
df_raw = load_raw(use_synthetic=True)
print(f'Raw shape: {df_raw.shape}')
display(df_raw.head())

In [ ]:
print('Missing values:')
print(df_raw.isnull().sum())
print('\nBasic statistics:')
display(df_raw.describe().round(3))

In [ ]:
# Distribution plots
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
cols = df_raw.columns[:6]
for ax, col in zip(axes.flat, cols):
    df_raw[col].dropna().hist(ax=ax, bins=50, color='#00e5a0', alpha=0.8)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
fig.suptitle('Feature Distributions', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 2. Clean & Feature Engineer

In [ ]:
df_clean  = clean(df_raw)
df_feat   = engineer_features(df_clean)
df_hourly = resample_hourly(df_feat)

print(f'After cleaning:    {df_clean.shape}')
print(f'After feat. eng.:  {df_feat.shape}')
print(f'After resampling:  {df_hourly.shape}')
display(df_hourly.head())

## 3. Visualise Consumption Trends

In [ ]:
target = df_hourly['Global_active_power']

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)

# Full time series
axes[0].plot(target.index, target.values, lw=1, color='#00e5a0', alpha=0.85)
axes[0].set_title('Hourly Energy Consumption (kW)')
axes[0].set_ylabel('kW')

# Rolling 24h mean
rolling = target.rolling(24).mean()
axes[1].plot(rolling.index, rolling.values, lw=1.5, color='#4da8ff')
axes[1].fill_between(rolling.index, rolling.values, alpha=0.2, color='#4da8ff')
axes[1].set_title('24-Hour Rolling Average (kW)')
axes[1].set_ylabel('kW')

# Average by hour-of-day
hourly_avg = target.groupby(target.index.hour).mean()
axes[2].bar(hourly_avg.index, hourly_avg.values, color='#ffb347', alpha=0.8)
axes[2].set_title('Average Consumption by Hour of Day')
axes[2].set_xlabel('Hour')
axes[2].set_ylabel('Avg kW')
axes[2].set_xticks(range(24))

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 7))
corr = df_hourly.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, linewidths=0.5,
    annot_kws={'size': 8}
)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 4. Train / Test Split & Normalisation

In [ ]:
from models.data_loader import get_processed_data

data = get_processed_data(look_back=24, use_synthetic=True)

print('Train sequences:', data['X_train_seq'].shape)
print('Test  sequences:', data['X_test_seq'].shape)
print('Features:       ', data['feature_cols'])

## 5. Train Linear Regression (Baseline)

In [ ]:
from models.linear_model import train_linear, predict_linear
from models.metrics import evaluate_predictions

model_lr = train_linear(data['X_train_flat'], data['y_train_flat'])
lr_preds = predict_linear(model_lr, data['X_test_flat'])
lr_metrics = evaluate_predictions(data['y_test_flat'], lr_preds, 'LinearRegression')

## 6. Train LSTM

In [ ]:
try:
    from models.lstm_model import train_lstm, predict_lstm
    X_tr  = data['X_train_seq']
    y_tr  = data['y_train_seq']
    val_n = int(len(X_tr) * 0.15)

    model_lstm, history = train_lstm(
        X_tr[:-val_n], y_tr[:-val_n],
        X_tr[-val_n:], y_tr[-val_n:],
        epochs=20, batch_size=64, patience=5
    )
    lstm_preds  = predict_lstm(model_lstm, data['X_test_seq'])
    lstm_metrics = evaluate_predictions(data['y_test_seq'], lstm_preds, 'LSTM')
    LSTM_TRAINED = True
except Exception as e:
    print(f'LSTM training skipped: {e}')
    LSTM_TRAINED = False
    history = {'loss': [], 'val_loss': []}

In [ ]:
# Training loss curve
if LSTM_TRAINED and history['loss']:
    plt.figure(figsize=(9, 4))
    plt.plot(history['loss'],     label='Train', lw=2, color='#00e5a0')
    plt.plot(history['val_loss'], label='Val',   lw=2, color='#ff4d6d', ls='--')
    plt.title('LSTM Training / Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Huber Loss')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No LSTM training history to plot.')

## 7. Compare Models

In [ ]:
from models.metrics import compare_models

results = {'LinearRegression': lr_metrics}
if LSTM_TRAINED:
    results['LSTM'] = lstm_metrics

compare_models(results)

In [ ]:
# Prediction vs Actual plot
n_plot = 150
y_true = data['y_test_seq'][:n_plot]
x_ax   = np.arange(n_plot)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(x_ax, y_true,   label='Actual',            lw=2,   color='#e2e8f0')
ax.plot(x_ax, lr_preds[:n_plot], label='Linear Reg', lw=1.5, color='#ff4d6d', ls='--')
if LSTM_TRAINED:
    ax.plot(x_ax, lstm_preds[:n_plot], label='LSTM', lw=1.5, color='#00e5a0', ls='-.')
ax.set_title('Prediction vs Actual – Test Set (first 150 samples)')
ax.set_xlabel('Time Step')
ax.set_ylabel('Normalised Energy')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 8. Anomaly Detection

In [ ]:
from models.anomaly_detector import detect_anomalies, generate_suggestions

# Inject synthetic spikes
series = df_hourly['Global_active_power'].values[-500:].copy()
series[[50, 120, 230, 380]] *= 3.0

result = detect_anomalies(series, method='rolling_zscore')
print(f"Anomalies found: {result['count']} ({result['pct']:.2f}%)  severity={result['severity']}")

# Plot
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(series, lw=1, color='#4da8ff', alpha=0.9, label='Consumption')
ax.scatter(result['indices'], [series[i] for i in result['indices']],
           color='#ff4d6d', zorder=5, s=60, label='Anomaly')
ax.set_title('Rolling Z-Score Anomaly Detection')
ax.set_xlabel('Time Step')
ax.set_ylabel('kW')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Optimisation suggestions
import pandas as pd
timestamps = pd.date_range('2023-01-01', periods=len(series), freq='h')
tips = generate_suggestions(series, timestamps=timestamps, anomaly_result=result)

for t in tips:
    print(f"\n[{t['category']}] {t['title']}")
    print(f"  Impact: {t['impact']} | Potential saving: {t['saving_pct']}%")
    print(f"  {t['detail'][:120]}…")